# 38. 조건 0개 쿼리는 무엇을 말하는가

노트북 37 에서 **설문 155건 중 75건(48.4%)이 accord 를 하나도 얻지 못했다.**

그 75건이 *"사전만 키우면 풀리는 것"* 인지 *"accord 로 바꿀 수 없는 것"* 인지 가른다.
전자면 사전 확장이 답이고, 후자면 `spec.md` §3 의 구조화 스키마에 칸이 없다는 더 큰 문제다.

## 0. 실행 조건과 한계

### holdout 취급 — 출력 형태를 제한한다

설문 155건은 `spec.md` §7.2 의 외부 holdout 이다.

**이 노트북은 유형별 분포까지만 낸다.** *"이 표현을 사전에 넣어라"* 는 후보 목록을
만들지 않는다. 만들면 holdout 으로 사전을 튜닝한 것이 되어 §7.2 의 역할이 깨진다.

예시 문장은 **유형마다 1~2개만** 보여준다. 판단에 필요한 최소한이다.

### 사전 노출 고백

분류 규칙을 짜기 전에 이미 본 것이 있다.

- 노트북 33 에서 쓴 가격·건강·개인화 정규식 3종
- 설문 원문 5건 (분포 확인 중 화면에 출력됨)

**그 3종은 그대로 쓰고 새로 추가한 유형은 원문을 보기 전에 정했다.**

### 그 밖

- **API 호출 0회.** 노트북 37 의 체크포인트를 읽는다
- 다중 라벨을 허용한다. 한 문장이 가격과 개인화를 동시에 말할 수 있다
- 키워드 규칙이므로 **재현 가능하지만 거칠다.** 표본 검수로 한계를 확인한다

In [1]:
import hashlib
import json
import pathlib
import re

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 90)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False
N_EXAMPLE = 2      # 유형당 보여줄 예시 문장 수 (holdout 노출 최소화)

## 1. 경로 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"

INPUT_PATHS = {
    "survey_ckpt": OUTPUT_DIR / "37_survey_stage1_checkpoint.csv",
    "survey_pq": OUTPUT_DIR / "37_survey_per_query.csv",
}
OUTPUT_PATHS = {
    "types": OUTPUT_DIR / "38_zero_condition_types.csv",
    "per_query": OUTPUT_DIR / "38_zero_condition_per_query.csv",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    d = hashlib.sha256()
    with pathlib.Path(path).open("rb") as h:
        for chunk in iter(lambda: h.read(1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
PROTECTED = {p.resolve() for p in INPUT_PATHS.values()}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
survey_ckpt,b72cbafc9d86a2ae
survey_pq,8c3c320cc61f1c69


## 2. 사전 등록 — 분류 기준을 원문 보기 전에 고정한다

두 축으로 나눈다.

**축 1 — 무엇을 요구했나** (다중 라벨)
**축 2 — accord 로 번역할 수 있는 종류인가** (축 1에서 기계적으로 결정)

In [3]:
# 축 1 — 요구 유형. 키워드는 결과를 보기 전에 고정했다.
TYPE_RULES = {
    "가격":       r"가격|가격대|비싸|저렴|만원|가성비|예산|싼 |부담",
    "건강·민감":  r"두통|민감|알레르기|임신|비염|어지러|자극적",
    "개인화":     r"나에게|나한테|저에게|저한테|내 스타일|제 스타일|내게|나와 |나랑",
    "장면·상황":  r"출근|회사|직장|데이트|소개팅|운동|학교|면접|결혼|하객|여행|일상|데일리|등교",
    "제품 특성":  r"지속력|잔향|발향|부향률|용량|니치|브랜드|오래 가|오래가|롱라스팅",
    "성별·나이":  r"남자|여자|남성|여성|[0-9]0대|나이|또래|중년|어려 보",
    "분위기·인상": r"느낌|무드|분위기|이미지|스타일|인상|매력|섹시|청순|단정|댄디|고급",
    "향 묘사":    r"향이|향을|향은|향수|냄새|시원|상큼|달콤|포근|깨끗|은은|진한|가벼운|부드러",
}

# 축 2 — 축 1 유형을 두 무리로 가른다.
TRANSLATABLE = {"분위기·인상", "향 묘사"}          # 사전을 키우면 accord 가 될 수 있다
NOT_TRANSLATABLE = {"가격", "건강·민감", "개인화", "제품 특성", "성별·나이"}
CONTEXT_ONLY = {"장면·상황"}                       # 스키마의 context 로 갈 수도 있다

PREREG = {
    "대상": "노트북 37 에서 c3 accord 가 0개였던 설문 쿼리",
    "축 1": "키워드 규칙 8종. 다중 라벨 허용",
    "축 2": "번역 가능(분위기·인상, 향 묘사) / 번역 불가(가격·건강·개인화·제품·성별) / 상황(장면)",
    "출력 제한": "유형별 분포와 유형당 예시 2건까지. 사전 후보 목록을 만들지 않는다",
    "한계": "키워드 규칙은 거칠다. 미분류 건수를 함께 보고한다",
}
display(pd.Series(PREREG, name="사전 등록").to_frame())

,사전 등록
대상,노트북 37 에서 c3 accord 가 0개였던 설문 쿼리
축 1,키워드 규칙 8종. 다중 라벨 허용
축 2,"번역 가능(분위기·인상, 향 묘사) / 번역 불가(가격·건강·개인화·제품·성별) / 상황(장면)"
출력 제한,유형별 분포와 유형당 예시 2건까지. 사전 후보 목록을 만들지 않는다
한계,키워드 규칙은 거칠다. 미분류 건수를 함께 보고한다


## 3. 대상 추출 — accord 0개 쿼리

In [4]:
ckpt = pd.read_csv(INPUT_PATHS["survey_ckpt"])
pq = pd.read_csv(INPUT_PATHS["survey_pq"])
d = ckpt.merge(pq[["query_id", "c1_n", "c2_n", "c3_n", "사전 표현", "글자 수"]],
               on="query_id", how="left")
zero = d[d["c3_n"] == 0].copy()
print(f"설문 {len(d)}건 중 accord 0개 {len(zero)}건 ({len(zero)/len(d):.1%})")

설문 155건 중 accord 0개 75건 (48.4%)


## 4. 먼저 확인 — 조건 0개가 '정보 0개' 는 아니다

accord 가 0개라는 것과 LLM 이 아무것도 못 뽑았다는 것은 다르다.
구조화 결과의 **다른 칸**이 채워졌는지 본다.

In [5]:
def fields(js):
    """구조화 JSON 의 필드별 채움 여부. dict."""
    o = json.loads(js) if isinstance(js, str) and js.strip() else {}
    c = o.get("context") or {}
    p = o.get("performance") or {}
    return {
        "scent": len(o.get("scent_preference") or []),
        "avoid": len(o.get("avoid") or []),
        "additional": len(o.get("additional_requirements") or []),
        "season": len(c.get("season") or []),
        "daypart": len(c.get("daypart") or []),
        "gender": len(c.get("gender") or []),
        "intensity": 1 if p.get("intensity") else 0,
        "longevity": 1 if p.get("longevity") else 0,
    }


f = pd.DataFrame([fields(j) for j in zero["parsed"]], index=zero.index)
zero = pd.concat([zero, f], axis=1)
display(pd.DataFrame({
    "채워진 쿼리 수": (f > 0).sum(),
    "비율": ((f > 0).mean()).round(3),
}))
empty_all = (f.sum(axis=1) == 0).sum()
print(f"\n모든 칸이 빈 쿼리: {empty_all}건")
print(f"scent 는 0인데 다른 칸이 채워진 쿼리: {len(zero) - empty_all}건")

,채워진 쿼리 수,비율
scent,4,0.053
avoid,9,0.120
additional,60,0.800
season,9,0.120
daypart,3,0.040
gender,15,0.200
intensity,8,0.107
longevity,4,0.053



모든 칸이 빈 쿼리: 12건
scent 는 0인데 다른 칸이 채워진 쿼리: 63건


### 왜 accord 가 0개인가 — 두 갈래

`scent_preference` 자체가 비었나, 아니면 **뽑히긴 했는데 accord 로 안 바뀌었나.**

In [6]:
zero["원인"] = [
    "scent 를 아예 못 뽑음" if s == 0 else "scent 는 뽑았으나 accord 로 안 바뀜"
    for s in zero["scent"]
]
display(zero["원인"].value_counts().to_frame("건수"))

,건수
원인,
scent 를 아예 못 뽑음,71
scent 는 뽑았으나 accord 로 안 바뀜,4


## 5. 축 1 — 무엇을 요구했나

In [7]:
def label_types(text):
    """문장에 걸리는 요구 유형들. list[str]."""
    t = str(text)
    return [k for k, pat in TYPE_RULES.items() if re.search(pat, t)]


zero["유형"] = zero["query_text"].map(label_types)
zero["유형 수"] = zero["유형"].map(len)

rows = []
for k in TYPE_RULES:
    hit = zero[zero["유형"].map(lambda v: k in v)]
    rows.append({"유형": k, "건수": len(hit), "비율": round(len(hit) / len(zero), 3)})
types = pd.DataFrame(rows).sort_values("건수", ascending=False)
display(types)
print(f"미분류(어떤 유형에도 안 걸림): {(zero['유형 수'] == 0).sum()}건")
print(f"유형 개수 평균: {zero['유형 수'].mean():.2f}")

,유형,건수,비율
7,향 묘사,61,0.813
6,분위기·인상,23,0.307
5,성별·나이,14,0.187
0,가격,10,0.133
2,개인화,6,0.080
3,장면·상황,5,0.067
4,제품 특성,4,0.053
1,건강·민감,1,0.013


미분류(어떤 유형에도 안 걸림): 13건
유형 개수 평균: 1.65


## 6. 축 2 — 사전을 키우면 풀리는가

In [8]:
def bucket(labels):
    """요구 유형 목록을 세 무리로 가른다. str."""
    s = set(labels)
    if s & TRANSLATABLE:
        return "번역 가능 — 향을 묘사했다"
    if s & CONTEXT_ONLY:
        return "상황만 — context 로 갈 수 있다"
    if s & NOT_TRANSLATABLE:
        return "번역 불가 — accord 와 무관"
    return "미분류"


zero["판정"] = zero["유형"].map(bucket)
b = zero["판정"].value_counts().to_frame("건수")
b["비율"] = (b["건수"] / len(zero)).round(3)
b["설문 155 대비"] = (b["건수"] / len(d)).round(3)
display(b)

,건수,비율,설문 155 대비
판정,,,
번역 가능 — 향을 묘사했다,62,0.827,0.400
미분류,13,0.173,0.084


### 읽는 법

**번역 가능** 은 향을 묘사했는데 사전에 그 표현이 없는 경우다. 사전을 키우면 준다.

**번역 불가** 는 가격·건강·개인화처럼 accord 로 바꿀 수 없는 요구다.
사전을 아무리 키워도 안 풀린다. `spec.md` §3 의 구조화 스키마에
`additional_requirements` 말고 받을 칸이 있는지가 따로 문제가 된다.

## 7. 유형별 예시 — 유형당 2건까지만

In [9]:
for k in types["유형"]:
    hit = zero[zero["유형"].map(lambda v: k in v)]
    if not len(hit):
        continue
    print(f"\n[{k}] {len(hit)}건")
    for t in hit["query_text"].head(N_EXAMPLE):
        print(f"   · {str(t)[:100]}")
un = zero[zero["유형 수"] == 0]
if len(un):
    print(f"\n[미분류] {len(un)}건")
    for t in un["query_text"].head(N_EXAMPLE * 2):
        print(f"   · {str(t)[:100]}")


[향 묘사] 61건
   · 내가 원하는 무드는 차분하고 댄디한 느낌을 원함. 이런 내 스타일에 맞춰서 향수를 추천해줘. 향이 오래갔으면 좋겠음
   · 향이 별로 안강했으면 좋겠어. 향에 민감해서 다른 향수 냄새를 맡으면 두통이 와. 그리고 자연스러운 향이 났으면 좋겠어. 너무 꾸민 티가 안나게. 그리고 가격대도 취준생 입장에서 

[분위기·인상] 23건
   · 내가 원하는 무드는 차분하고 댄디한 느낌을 원함. 이런 내 스타일에 맞춰서 향수를 추천해줘. 향이 오래갔으면 좋겠음
   · 나에게 어울리는 향수를 추천해줘 나는 자연 스럽고 산뜻한 느낌을 원해

[성별·나이] 14건
   · 운동하고 뿌리면 좋은 비싸지 않은 남자 향수 추천해줘.
   · 플로럴? 꽃 냄새는 별로고 좀 포근하고 딱 맡았을 때 와 진짜 어른 여자<< 같은 느낌이 되고싶은데 어떤 향수가 좋을까요?

[가격] 10건
   · 향이 별로 안강했으면 좋겠어. 향에 민감해서 다른 향수 냄새를 맡으면 두통이 와. 그리고 자연스러운 향이 났으면 좋겠어. 너무 꾸민 티가 안나게. 그리고 가격대도 취준생 입장에서 
   · 운동하고 뿌리면 좋은 비싸지 않은 남자 향수 추천해줘.

[개인화] 6건
   · 내가 원하는 무드는 차분하고 댄디한 느낌을 원함. 이런 내 스타일에 맞춰서 향수를 추천해줘. 향이 오래갔으면 좋겠음
   · 나에게 어울리는 향수를 추천해줘 나는 자연 스럽고 산뜻한 느낌을 원해

[장면·상황] 5건
   · 운동하고 뿌리면 좋은 비싸지 않은 남자 향수 추천해줘.
   · 이 편지는 영국에서 최초로 시작되어 일년에 한바퀴를 돌면서 받는 사람에게 행운을 주었고 지금은 당신에게로 옮겨진 이 편지는 4일 안에 당신 곁을 떠나야 합니다. 이 편지를 포함해서

[제품 특성] 4건
   · 여름이니까 가벼운 향을 추천받고 싶어. 약간 비누향도 좋고. 여성스러운 느낌이면좋겠어. 가격은 5만원 아래로. 향이 오래 가면 좋겠다.
   · 내가 자주 사용하는 모스키노 토이2 EDP 향수가 있는데, 

## 8. 표본 검수 — 키워드 규칙이 얼마나 거친가

규칙이 놓치거나 잘못 붙인 것을 확인한다. **결과를 보고 규칙을 고치지 않는다** —
고치면 사전 등록이 깨지고 holdout 튜닝이 된다. 한계로 기록만 한다.

In [10]:
sample = zero.sample(n=min(10, len(zero)), random_state=42)
display(sample[["query_text", "유형", "판정"]].reset_index(drop=True))

,query_text,유형,판정
0,바디워시느낌이 나는 향수를 추천해주세요,"[분위기·인상, 향 묘사]",번역 가능 — 향을 묘사했다
1,네.,[],미분류
2,"나 향수를 잘 모르는데, 깔끔한 스타일의 향을 필요로 해. 최대 15만원 안팎","[가격, 분위기·인상, 향 묘사]",번역 가능 — 향을 묘사했다
3,내가 원하는 무드는 차분하고 댄디한 느낌을 원함. 이런 내 스타일에 맞춰서 향수를 추천해줘. 향이 오래갔으면 좋겠음,"[개인화, 분위기·인상, 향 묘사]",번역 가능 — 향을 묘사했다
4,예,[],미분류
5,가볍고 향 맡으면 머리 안아픈 향을 추천해줘,[향 묘사],번역 가능 — 향을 묘사했다
6,"내가 주로 사용했던 향수는 베르사체 딜런퍼플, 아쿠아 디파르마, 르라보 모하비고스트, 티파니앤코 러브포힘,딥디크 플레르드뽀 인데 비슷한 향을 가진 향수 추...",[향 묘사],번역 가능 — 향을 묘사했다
7,내가 쓰는 향수랑 비슷한 향이 나는 다른 향수가 궁금하다,[향 묘사],번역 가능 — 향을 묘사했다
8,향이 강하지 않으면서 고급스럽고 오래 지속되는 향,"[분위기·인상, 향 묘사]",번역 가능 — 향을 묘사했다
9,예,[],미분류


## 9. 저장

In [11]:
out = zero[["query_id", "글자 수", "scent", "additional", "season", "daypart",
            "gender", "intensity", "longevity", "원인", "판정"]].copy()
out["유형"] = zero["유형"].map(lambda v: "|".join(v))
write_output(OUTPUT_PATHS["types"], lambda p: types.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["per_query"],
             lambda p: out.to_csv(p, index=False, encoding="utf-8-sig"))
print("\n원문(query_text)은 저장하지 않는다 — holdout 노출을 늘리지 않기 위해서다.")

저장: analysis_outputs\38_zero_condition_types.csv
저장: analysis_outputs\38_zero_condition_per_query.csv

원문(query_text)은 저장하지 않는다 — holdout 노출을 늘리지 않기 위해서다.


## 10. 가드 검증

In [12]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in after if after[k] != input_hashes_before[k]]
if changed:
    raise RuntimeError(f"입력 파일이 변경됐다: {changed}")
print("입력 해시 불변 확인:", ", ".join(INPUT_PATHS))
print()
print("주의 — 여기서 본 표현을 사전에 넣으면 설문이 holdout 이 아니게 된다.")

입력 해시 불변 확인: survey_ckpt, survey_pq

주의 — 여기서 본 표현을 사전에 넣으면 설문이 holdout 이 아니게 된다.
